# E8 prior study: PEM with a chosen prior, then the E8 graphs (Kaggle, CPU)

Runs the **full PEM** (all 2,056 units) on the VAR training residuals with one prior, then builds the **E8 graphs**
(`S_h = P ⊙ |Ψ_h|`, top-25, h = 1, 3, 6, 12). Nothing else changes between runs, so the only difference between two
versions of this notebook is the prior. **No LiNGAM, no B0**: `lingam_result.npz` is not needed and is not linked.

Priors (`PRIOR` in the settings): **`flow`** (proposed): Pearson correlation of the ORIGINAL detrended training flow,
η = 0.05 + 0.90 |r| on the pairs sharing a causal unit (0.5 elsewhere); `uniform` (η = 0.5, the baseline); `pearson`,
`spearman`, `partial` (correlation of the VAR residuals, earlier study). The VAR residuals are used only as the data
PEM estimates the precision matrix from. The flow prior needs the GBA traffic file (`gba_his_raw_2021.h5`), which is in
the GBA dataset input. PEM settings are fixed for all of them: ν0 = 0.01, ν1 = 0.2 (a graded P;
the original 0.0017 / 1 leaves P at 0 or 1, so no prior can matter).

**Before running** (Add Input)
1. The GBA dataset (`gba_rn_adj.npy`, and `gba_his_raw_2021.h5` for the flow prior).
2. The **PEM inputs dataset** (`pem-inputs`): `residuals.npy`, `metadata.json`, `sensors.npy`, `coefficients.npy`.
3. Settings: **Accelerator = None (CPU)**, **Internet = On**. Run one version per prior: Save Version -> Save & Run All.
4. If PEM stops at the time limit: open a new version, *Add Input* -> *Notebook Output* of this notebook, run again;
   finished units are copied back and skipped.
5. To compare a correlation prior with the uniform prior inside the run, also add the output of the uniform version
   as an input (its `uniform_nu0_0.01_nu1_0.2__pem_result.npz` is found automatically).

Download `e8_prior_output/` and upload the `E8__h*__<variant>.npz` files to the artifacts dataset. Train them in
`ablation-study.ipynb` with `EXPERIMENT = "E8"` and `GRAPH_VARIANT = "<variant>"`.

## Settings

In [ ]:
import os

PRIOR   = "flow"           # "flow" (proposed) | "flow_shuffle" (control) | "uniform" (baseline) | "pearson" | "spearman" | "partial" (residual correlation, earlier study)
NU0     = 0.01             # fixed for the whole prior study
NU1     = 0.2
HORIZONS = [1, 3, 6, 12]
TOP_K    = 25

REPO_URL    = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH = "main"
WORKERS            = os.cpu_count()
TIME_BUDGET_HOURS  = 11.0

assert PRIOR in ("flow", "flow_shuffle", "uniform", "pearson", "spearman", "partial")
TAG     = f"e8prior_{PRIOR}_nu0_{NU0:g}_nu1_{NU1:g}"
VARIANT = f"{PRIOR}_nu0_{NU0:g}_nu1_{NU1:g}"
print("prior:", PRIOR, "| PEM nu0/nu1:", NU0, NU1, "| variant:", VARIANT)

## 1. Code and inputs

In [ ]:
import glob, json, shutil, subprocess, sys, time
import numpy as np

WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
for needed in ("build_corr_prior.py", "causal/corr_prior.py", "build_e8_graphs.py"):
    assert os.path.exists(f"{CODE}/{needed}"), f"{needed} is not on GitHub yet: push the prior-study files first."
assert "--prior-file" in open(f"{CODE}/run_local_pem.py").read(), "push the updated run_local_pem.py first."
assert "--variant" in open(f"{CODE}/build_e8_graphs.py").read(), "push the updated build_e8_graphs.py first."


def find_one(name):
    found = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True)
             if "/units/" not in p and "/implementation/" not in p]
    assert len(found) == 1, f"expected exactly one {name} under /kaggle/input, found {found}"
    return found[0]


os.makedirs(f"{CODE}/datasets", exist_ok=True)
os.symlink(os.path.dirname(find_one("gba_rn_adj.npy")), f"{CODE}/datasets/LargeST GBA")

VAR_DIR = f"{CODE}/outputs/var_train/lag3_alpha1e+06"
os.makedirs(VAR_DIR, exist_ok=True)
for name in ("residuals.npy", "metadata.json", "sensors.npy", "coefficients.npy"):
    os.symlink(find_one(name), f"{VAR_DIR}/{name}")        # symlinks: nothing large is copied

sys.path.insert(0, CODE)
from config import config as C
assert (C.VAR_LAG, C.VAR_ALPHA) == (3, 1e6) and C.PEM_STANDARDISE and C.PEM_OPTIMIZER == "L-BFGS-B"
assert (C.PEM_TAU, C.PEM_THRESHOLD) == (0.01, 0.5)
print("VAR run:", C.VAR_RUN, "| files:", sorted(os.listdir(VAR_DIR)))

## 2. The prior

In [ ]:
PRIOR_FILE = None
if PRIOR != "uniform":
    args = {"flow": ["--source", "flow"], "flow_shuffle": ["--source", "flow", "--shuffle-seed", "42"]}.get(PRIOR, ["--method", PRIOR])
    r = subprocess.run([sys.executable, "build_corr_prior.py", *args], cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-1500:])
    assert r.returncode == 0
    PRIOR_FILE = f"{CODE}/outputs/priors/corr_{ {'flow_shuffle': 'flow_shuffle42'}.get(PRIOR, PRIOR) }.npz"
print("prior file:", PRIOR_FILE)

## 3. Resume finished units (if an earlier version's output is added as input)

In [ ]:
PEM_DIR  = f"{CODE}/outputs/pem/{TAG}"
UNIT_DIR = f"{PEM_DIR}/units"
os.makedirs(UNIT_DIR, exist_ok=True)
previous = [p for p in glob.glob("/kaggle/input/**/unit_*.npz", recursive=True) if f"e8prior_units__{TAG}" in p]
for p in previous:
    shutil.copy(p, UNIT_DIR)
print("units copied from an earlier run:", len(previous))

## 4. Run PEM (time budget)

In [ ]:
cmd = [sys.executable, "-u", "run_local_pem.py", "--nu0", str(NU0), "--nu1", str(NU1), "--tag", TAG,
       "--workers", str(WORKERS)]
if PRIOR_FILE:
    cmd += ["--prior-file", PRIOR_FILE]
print(" ".join(cmd))

start = time.time()
log = open(f"{WORK}/pem_{TAG}.log", "w")
proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
last = 0
for line in proc.stdout:
    log.write(line); log.flush()
    if time.time() - last > 300 or "error" in line.lower() or "saved" in line.lower():
        print(f"[{(time.time() - start) / 3600:5.2f} h] {line.rstrip()}", flush=True)
        last = time.time()
    if time.time() - start > TIME_BUDGET_HOURS * 3600:
        proc.kill()
        print("Time budget reached: PEM stopped; finished units are kept.")
        break
proc.wait()
log.close()

PEM_COMPLETE = os.path.exists(f"{PEM_DIR}/pem_result.npz")
print(f"units finished: {len(glob.glob(UNIT_DIR + '/unit_*.npz'))} / 2056 | pem_result.npz written: {PEM_COMPLETE} "
      f"| hours: {(time.time() - start) / 3600:.2f}")

## 5. How graded is P, and what did the prior change?

In [ ]:
STATS = {}
if PEM_COMPLETE:
    with np.load(f"{PEM_DIR}/pem_result.npz") as r:
        P = r["inclusion_probabilities"].astype(np.float64)
        print("prior recorded in the result:", str(r["prior"]))
    off = ~np.eye(len(P), dtype=bool)
    cand = (P > 0) & off          # pairs PEM estimated (sharing a unit)
    v = P[cand]
    STATS = {"prior": PRIOR, "nu0": NU0, "nu1": NU1, "entries_estimated": int(v.size),
             "share_P_between_0.05_and_0.95": float(((v > 0.05) & (v < 0.95)).mean()),
             "share_P_gt_0.99": float((v > 0.99).mean()), "share_P_lt_0.01": float((v < 0.01).mean()),
             "share_P_ge_0.5": float((v >= 0.5).mean())}
    # optional comparison with the uniform-prior run of the same settings (add its output as an input)
    others = glob.glob(f"/kaggle/input/**/uniform_nu0_{NU0:g}_nu1_{NU1:g}__pem_result.npz", recursive=True)
    if PRIOR != "uniform" and others:
        with np.load(others[0]) as r:
            U = r["inclusion_probabilities"].astype(np.float64)
        from scipy.stats import spearmanr
        d = np.abs(P - U)[cand]
        STATS["vs_uniform"] = {"file": others[0], "share_changed_over_0.01": float((d > 0.01).mean()),
                               "share_changed_over_0.10": float((d > 0.10).mean()),
                               "share_crossing_0.5": float(((P >= 0.5) != (U >= 0.5))[cand].mean()),
                               "rank_corr_with_uniform_P": float(spearmanr(P[cand], U[cand])[0])}
    print(json.dumps(STATS, indent=2))
else:
    print("PEM not finished: run a new version with this version's output added as input (section 3).")

## 6. Build the E8 graphs (no LiNGAM; only when PEM finished)

In [ ]:
OUT = f"{WORK}/e8_prior_output"
os.makedirs(OUT, exist_ok=True)
if PEM_COMPLETE:
    cmd = [sys.executable, "-u", "build_e8_graphs.py", "--pem-result", f"{PEM_DIR}/pem_result.npz",
           "--variant", VARIANT, "--graph-dir", f"{OUT}/graphs", "--out-dir", f"{OUT}/e8",
           "--k", str(TOP_K), "--horizons", *map(str, HORIZONS)]
    r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-5000:], r.stderr[-3000:])
    assert r.returncode == 0
    json.dump(STATS, open(f"{OUT}/prior_stats_{VARIANT}.json", "w"), indent=2)

## 7. Outputs and clean up

In [ ]:
if PEM_COMPLETE:
    for p in (f"{PEM_DIR}/pem_result.npz", f"{PEM_DIR}/summary.json", f"{WORK}/pem_{TAG}.log"):
        if os.path.exists(p):
            shutil.copy(p, f"{OUT}/{VARIANT}__{os.path.basename(p)}")
else:
    shutil.copytree(UNIT_DIR, f"{WORK}/e8prior_units__{TAG}", dirs_exist_ok=True)
shutil.rmtree(CODE)
for root, _, files in os.walk(WORK):
    for f in sorted(files):
        pth = os.path.join(root, f)
        print(f"{os.path.getsize(pth) / 1e6:8.2f} MB  {os.path.relpath(pth, WORK)}")
print("Upload graphs/E8__h*__" + VARIANT + ".npz to the artifacts dataset; train with ablation-study.ipynb "
      "(EXPERIMENT = 'E8', GRAPH_VARIANT = '" + VARIANT + "').")